In [1]:
import numpy as np
import torch
import dgl

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("dgl:", dgl.__version__)

numpy: 1.26.4
torch: 2.0.1+cpu
dgl: 1.1.2


In [11]:
# ============================================================
# Graph Anomaly Detection datasets downloader / organizer
# Saves everything near the current Jupyter notebook
# ============================================================

from pathlib import Path
import os
import shutil
import urllib.request
import gzip
import pandas as pd

ROOT = Path.cwd()
DATA_ROOT = ROOT / "graph_anomaly_datasets"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Current folder:", ROOT)
print("Dataset root:", DATA_ROOT)

Current folder: C:\Users\ivan\WORK\workshops\ICDM\Ablation
Dataset root: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets


In [12]:
# ============================================================
# Helpers
# ============================================================

def make_dir(name: str) -> Path:
    path = DATA_ROOT / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def download_file(url: str, out_path: Path, overwrite: bool = False):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not overwrite:
        print(f"[SKIP] Already exists: {out_path}")
        return out_path

    print(f"[DOWNLOAD] {url}")
    print(f"          -> {out_path}")
    urllib.request.urlretrieve(url, out_path)
    print("[OK]")
    return out_path


def read_gzip_csv(path: Path, names=None):
    return pd.read_csv(path, compression="gzip", names=names)


def save_dataframe(df: pd.DataFrame, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path} shape={df.shape}")

In [3]:
# ============================================================
# Bitcoin-Alpha / Bitcoin-OTC from SNAP
# ============================================================

bitcoin_dir = make_dir("bitcoin_snap")

bitcoin_alpha_url = "https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz"
bitcoin_otc_url = "https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz"

alpha_gz = download_file(
    bitcoin_alpha_url,
    bitcoin_dir / "soc-sign-bitcoinalpha.csv.gz"
)

otc_gz = download_file(
    bitcoin_otc_url,
    bitcoin_dir / "soc-sign-bitcoinotc.csv.gz"
)

columns = ["source", "target", "rating", "timestamp"]

alpha_df = read_gzip_csv(alpha_gz, names=columns)
otc_df = read_gzip_csv(otc_gz, names=columns)

save_dataframe(alpha_df, bitcoin_dir / "bitcoin_alpha_edges.csv")
save_dataframe(otc_df, bitcoin_dir / "bitcoin_otc_edges.csv")

print(alpha_df.head())
print(otc_df.head())

[DOWNLOAD] https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz
          -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\soc-sign-bitcoinalpha.csv.gz
[OK]
[DOWNLOAD] https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz
          -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\soc-sign-bitcoinotc.csv.gz
[OK]
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_alpha_edges.csv shape=(24186, 4)
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_otc_edges.csv shape=(35592, 4)
   source  target  rating   timestamp
0    7188       1      10  1407470400
1     430       1      10  1376539200
2    3134       1      10  1369713600
3    3026       1      10  1350014400
4    3010       1      10  1347854400
   source  target  rating     timestamp
0       6       2       4  1.289242e+09
1       6       5       2  1.289242e+09
2       

In [6]:
# ============================================================
# DGL datasets
# FraudYelp, FraudAmazon, BitcoinOTC
# ============================================================

dgl_dir = make_dir("dgl_raw")

try:
    import dgl
    from dgl.data import FraudYelpDataset, FraudAmazonDataset, BitcoinOTCDataset

    print("DGL version:", dgl.__version__)

    # DGL will save its internal files here
    os.environ["DGLDEFAULTDIR"] = str(dgl_dir)

    print("\n[LOAD] FraudYelpDataset")
    fraud_yelp = FraudYelpDataset(raw_dir=str(dgl_dir / "fraud_yelp"))
    gy = fraud_yelp[0]
    print(gy)

    print("\n[LOAD] FraudAmazonDataset")
    fraud_amazon = FraudAmazonDataset(raw_dir=str(dgl_dir / "fraud_amazon"))
    ga = fraud_amazon[0]
    print(ga)

    print("\n[LOAD] BitcoinOTCDataset")
    bitcoin_otc_dgl = BitcoinOTCDataset(raw_dir=str(dgl_dir / "bitcoin_otc_dgl"))
    gotc = bitcoin_otc_dgl[0]
    print(gotc)

except Exception as e:
    print("[ERROR] DGL loading failed.")
    print("Try installing DGL:")
    print("!pip install dgl -q")
    print("Error:", repr(e))

DGL version: 1.1.2

[LOAD] FraudYelpDataset
Extracting file to C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\dgl_raw\fraud_yelp\yelp_a7a80596
Done saving data into cached files.
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])

[LOAD] FraudAmazonDataset
Extracting file to C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\dgl_raw\fraud_amazon\amazon_1e446145
Done saving data into cached files.
Graph(num_nodes={'user': 11944},
      num_edges={('user', 'net_upu', 'user'): 351216, ('user', 'net_usu', 'user'): 7132958, ('user', 'net_uvu', 'user'): 2073474},
      metagraph=[('user', 'user', 'net_upu'), ('user', 'user', 'net_usu'), ('user', 'user', 'net_uvu')])

[LOAD] BitcoinOTCDataset
Extracting file to C:\Users\

In [7]:
# ============================================================
# Save DGL graphs into local folders
# ============================================================

try:
    from dgl.data.utils import save_graphs

    processed_dgl_dir = make_dir("dgl_processed")

    save_graphs(str(processed_dgl_dir / "fraud_yelp.bin"), [gy])
    save_graphs(str(processed_dgl_dir / "fraud_amazon.bin"), [ga])
    save_graphs(str(processed_dgl_dir / "bitcoin_otc_dgl_snapshot0.bin"), [gotc])

    print("[SAVED] DGL processed graphs:")
    print(processed_dgl_dir)

except Exception as e:
    print("[ERROR] Could not save DGL graphs.")
    print(repr(e))

[SAVED] DGL processed graphs:
C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\dgl_processed


In [8]:
# ============================================================
# Export DGL node features and labels
# ============================================================

import torch
import numpy as np

def export_dgl_graph_data(g, out_dir: Path, name: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[EXPORT] {name}")
    print(g)

    # Save full graph object
    try:
        from dgl.data.utils import save_graphs
        save_graphs(str(out_dir / f"{name}.bin"), [g])
    except Exception as e:
        print("Could not save DGL graph:", repr(e))

    # Save node data
    for key, value in g.ndata.items():
        out_path = out_dir / f"{name}_ndata_{key}.pt"
        torch.save(value, out_path)
        print(f"Saved node data: {key} -> {out_path} shape={tuple(value.shape)}")

    # Save edge data
    for key, value in g.edata.items():
        out_path = out_dir / f"{name}_edata_{key}.pt"
        torch.save(value, out_path)
        print(f"Saved edge data: {key} -> {out_path} shape={tuple(value.shape)}")

    # Save edges as CSV
    src, dst = g.edges()
    edges_df = pd.DataFrame({
        "source": src.cpu().numpy(),
        "target": dst.cpu().numpy(),
    })
    edges_df.to_csv(out_dir / f"{name}_edges.csv", index=False)
    print(f"Saved edges CSV: {out_dir / f'{name}_edges.csv'}")


try:
    export_dir = make_dir("exports")

    export_dgl_graph_data(gy, export_dir / "fraud_yelp", "fraud_yelp")
    export_dgl_graph_data(ga, export_dir / "fraud_amazon", "fraud_amazon")
    export_dgl_graph_data(gotc, export_dir / "bitcoin_otc_dgl", "bitcoin_otc_dgl")

except Exception as e:
    print("[ERROR] Export failed.")
    print(repr(e))


[EXPORT] fraud_yelp
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
Saved node data: feature -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp_ndata_feature.pt shape=(45954, 32)
Saved node data: label -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp_ndata_label.pt shape=(45954,)
Saved node data: train_mask -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp_ndata_train_mask.pt shape=(45954,)
Saved node data: val_mask -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp_ndata_val_mask.pt shape=(45954,)
Saved node data: test_mask -> C:\Users\ivan\WORK\works

In [9]:
from pathlib import Path
import pandas as pd
import torch

def safe_name(x):
    return str(x).replace("(", "").replace(")", "").replace("'", "").replace(",", "_").replace(" ", "")

def export_dgl_graph_data(g, out_dir: Path, name: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[EXPORT] {name}")
    print(g)

    # Save full graph object
    try:
        from dgl.data.utils import save_graphs
        save_graphs(str(out_dir / f"{name}.bin"), [g])
        print(f"Saved graph: {out_dir / f'{name}.bin'}")
    except Exception as e:
        print("Could not save DGL graph:", repr(e))

    # Save node data
    for ntype in g.ntypes:
        ntype_dir = out_dir / f"nodes_{ntype}"
        ntype_dir.mkdir(parents=True, exist_ok=True)

        for key, value in g.nodes[ntype].data.items():
            out_path = ntype_dir / f"{name}_{ntype}_ndata_{key}.pt"
            torch.save(value, out_path)
            print(f"Saved node data: ntype={ntype}, key={key} -> {out_path} shape={tuple(value.shape)}")

    # Save edge data and edge lists by edge type
    for canonical_etype in g.canonical_etypes:
        src_type, etype, dst_type = canonical_etype
        etype_name = safe_name("_".join(canonical_etype))

        etype_dir = out_dir / f"edges_{etype_name}"
        etype_dir.mkdir(parents=True, exist_ok=True)

        src, dst = g.edges(etype=canonical_etype)

        edges_df = pd.DataFrame({
            "source": src.cpu().numpy(),
            "target": dst.cpu().numpy(),
            "src_type": src_type,
            "edge_type": etype,
            "dst_type": dst_type,
        })

        edges_path = etype_dir / f"{name}_{etype_name}_edges.csv"
        edges_df.to_csv(edges_path, index=False)
        print(f"Saved edges: {canonical_etype} -> {edges_path} shape={edges_df.shape}")

        for key, value in g.edges[canonical_etype].data.items():
            out_path = etype_dir / f"{name}_{etype_name}_edata_{key}.pt"
            torch.save(value, out_path)
            print(f"Saved edge data: etype={canonical_etype}, key={key} -> {out_path} shape={tuple(value.shape)}")

In [10]:
export_dir = DATA_ROOT / "exports"

export_dgl_graph_data(gy, export_dir / "fraud_yelp", "fraud_yelp")
export_dgl_graph_data(ga, export_dir / "fraud_amazon", "fraud_amazon")


[EXPORT] fraud_yelp
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
Saved graph: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp.bin
Saved node data: ntype=review, key=feature -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_feature.pt shape=(45954, 32)
Saved node data: ntype=review, key=label -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_label.pt shape=(45954,)
Saved node data: ntype=review, key=train_mask -> C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_train_mask.pt 

In [2]:
from pathlib import Path
import torch
from torch_geometric.datasets import Reddit

ROOT = Path.cwd()
DATA_ROOT = ROOT / "graph_anomaly_datasets"

reddit_dir = DATA_ROOT / "pyg" / "reddit"
reddit_dir.mkdir(parents=True, exist_ok=True)

reddit_dataset = Reddit(root=str(reddit_dir))
reddit_data = reddit_dataset[0]

print(reddit_data)
print("x:", reddit_data.x.shape)
print("edge_index:", reddit_data.edge_index.shape)
print("y:", reddit_data.y.shape)

torch.save(reddit_data, reddit_dir / "reddit_pyg_data.pt")
print("[SAVED]", reddit_dir / "reddit_pyg_data.pt")

Extracting C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\reddit\raw\reddit.zip
Processing...
Done!


Data(x=[232965, 602], edge_index=[2, 114615892], y=[232965], train_mask=[232965], val_mask=[232965], test_mask=[232965])
x: torch.Size([232965, 602])
edge_index: torch.Size([2, 114615892])
y: torch.Size([232965])
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\reddit\reddit_pyg_data.pt


In [4]:
from pathlib import Path
import torch

ROOT = Path.cwd()
DATA_ROOT = ROOT / "graph_anomaly_datasets"
PYG_DIR = DATA_ROOT / "pyg"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
PYG_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("PYG_DIR:", PYG_DIR)

ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation
DATA_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets
PYG_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg


In [6]:
from pathlib import Path
import torch

try:
    from torch_geometric.datasets import DGraphFin

    dgraph_dir = PYG_DIR / "dgraph_fin"
    dgraph_dir.mkdir(parents=True, exist_ok=True)

    dgraph_dataset = DGraphFin(root=str(dgraph_dir))
    dgraph_data = dgraph_dataset[0]

    print(dgraph_data)
    print("x:", dgraph_data.x.shape if hasattr(dgraph_data, "x") and dgraph_data.x is not None else None)
    print("edge_index:", dgraph_data.edge_index.shape)
    print("y:", dgraph_data.y.shape if hasattr(dgraph_data, "y") and dgraph_data.y is not None else None)

    torch.save(dgraph_data, dgraph_dir / "dgraph_fin_pyg_data.pt")
    print("[SAVED]", dgraph_dir / "dgraph_fin_pyg_data.pt")

except Exception as e:
    dgraph_dir = PYG_DIR / "dgraph_fin"
    dgraph_dir.mkdir(parents=True, exist_ok=True)

    print("[WARNING] DGraph-Fin was not downloaded automatically.")
    print("This dataset often requires manual download/registration.")
    print("Folder prepared here:")
    print(dgraph_dir)
    print("Put the downloaded files there and rerun this cell.")
    print("Error:", repr(e))

[WARNING] DGraph-Fin was not downloaded automatically.
This dataset often requires manual download/registration.
Folder prepared here:
C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\dgraph_fin
Put the downloaded files there and rerun this cell.
Error: RuntimeError("Dataset not found. Please download 'DGraphFin.zip' from 'https://dgraph.xinye.com' and move it to 'C:\\Users\\ivan\\WORK\\workshops\\ICDM\\Ablation\\graph_anomaly_datasets\\pyg\\dgraph_fin\\raw'")


In [7]:
from pathlib import Path

zip_path = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\dgraph_fin\raw\DGraphFin.zip")

print("exists:", zip_path.exists())
print("size MB:", zip_path.stat().st_size / 1024 / 1024 if zip_path.exists() else None)

exists: True
size MB: 143.50540161132812


In [8]:
from pathlib import Path
import torch
from torch_geometric.datasets import DGraphFin

dgraph_dir = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\dgraph_fin")

dgraph_dataset = DGraphFin(root=str(dgraph_dir))
dgraph_data = dgraph_dataset[0]

print(dgraph_data)
print("x:", dgraph_data.x.shape if dgraph_data.x is not None else None)
print("edge_index:", dgraph_data.edge_index.shape)
print("y:", dgraph_data.y.shape if dgraph_data.y is not None else None)

torch.save(dgraph_data, dgraph_dir / "dgraph_fin_pyg_data.pt")
print("[SAVED]", dgraph_dir / "dgraph_fin_pyg_data.pt")

Processing...
Done!


Data(x=[3700550, 17], edge_index=[2, 4300999], y=[3700550], edge_type=[4300999], edge_time=[4300999], train_mask=[3700550], val_mask=[3700550], test_mask=[3700550])
x: torch.Size([3700550, 17])
edge_index: torch.Size([2, 4300999])
y: torch.Size([3700550])
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\pyg\dgraph_fin\dgraph_fin_pyg_data.pt


In [13]:
# ============================================================
# Clone useful benchmark repositories
# ============================================================

repos_dir = make_dir("repos")

repos = {
    "GADBench": "https://github.com/squareRoot3/GADBench.git",
    "BWGNN_Rethinking_Anomaly_Detection": "https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git",
}

for repo_name, repo_url in repos.items():
    target = repos_dir / repo_name

    if target.exists():
        print(f"[SKIP] Repo already exists: {target}")
    else:
        cmd = f'git clone "{repo_url}" "{target}"'
        print("[RUN]", cmd)
        os.system(cmd)

[RUN] git clone "https://github.com/squareRoot3/GADBench.git" "C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\repos\GADBench"
[RUN] git clone "https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git" "C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\repos\BWGNN_Rethinking_Anomaly_Detection"


In [19]:
from pathlib import Path
import os
import json

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets")

print("=" * 120)
print("GRAPH ANOMALY DATASETS AUDIT")
print("=" * 120)
print("DATA_ROOT:", DATA_ROOT)
print("EXISTS:", DATA_ROOT.exists())
print()

def size_mb(path: Path) -> float:
    if path.is_file():
        return path.stat().st_size / 1024 / 1024
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / 1024 / 1024

def count_files(path: Path) -> int:
    if path.is_file():
        return 1
    return sum(1 for p in path.rglob("*") if p.is_file())

def show_tree(path: Path, max_depth=3, prefix="", depth=0):
    if depth > max_depth:
        return

    if not path.exists():
        return

    items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))

    for i, item in enumerate(items):
        connector = "└── " if i == len(items) - 1 else "├── "
        if item.is_dir():
            n_files = count_files(item)
            mb = size_mb(item)
            print(f"{prefix}{connector}{item.name}/  [{n_files} files, {mb:.2f} MB]")
            extension = "    " if i == len(items) - 1 else "│   "
            show_tree(item, max_depth=max_depth, prefix=prefix + extension, depth=depth + 1)
        else:
            mb = size_mb(item)
            print(f"{prefix}{connector}{item.name}  [{mb:.2f} MB]")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Not found: {DATA_ROOT}")

print("TOP-LEVEL SUMMARY")
print("-" * 120)

rows = []
for p in sorted(DATA_ROOT.iterdir(), key=lambda x: x.name.lower()):
    rows.append({
        "name": p.name,
        "type": "dir" if p.is_dir() else "file",
        "files": count_files(p),
        "size_mb": round(size_mb(p), 2),
        "path": str(p),
    })

for r in rows:
    print(f"{r['name']:<35} {r['type']:<5} files={r['files']:<8} size={r['size_mb']:>10.2f} MB")

print()
print("=" * 120)
print("TREE, DEPTH=3")
print("=" * 120)
show_tree(DATA_ROOT, max_depth=3)

print()
print("=" * 120)
print("EXPECTED DATASET CHECKS")
print("=" * 120)

checks = {
    "Bitcoin Alpha SNAP CSV": DATA_ROOT / "bitcoin_snap" / "bitcoin_alpha_edges.csv",
    "Bitcoin OTC SNAP CSV": DATA_ROOT / "bitcoin_snap" / "bitcoin_otc_edges.csv",

    "FraudYelp graph bin": DATA_ROOT / "exports" / "fraud_yelp" / "fraud_yelp.bin",
    "FraudYelp features": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_feature.pt",
    "FraudYelp labels": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_label.pt",

    "FraudAmazon graph bin": DATA_ROOT / "exports" / "fraud_amazon" / "fraud_amazon.bin",
    "FraudAmazon features": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_review" / "fraud_amazon_review_ndata_feature.pt",
    "FraudAmazon labels": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_review" / "fraud_amazon_review_ndata_label.pt",

    "DGL Bitcoin OTC graph bin": DATA_ROOT / "exports" / "bitcoin_otc_dgl" / "bitcoin_otc_dgl.bin",

    "PyG Reddit data": DATA_ROOT / "pyg_reddit" / "reddit_pyg_data.pt",

    "GADBench repo": DATA_ROOT / "repos" / "GADBench",
    "DGraphFin zip": DATA_ROOT / "pyg_dgraph_fin" / "raw" / "DGraphFin.zip",
    "DGraphFin processed pt": DATA_ROOT / "pyg_dgraph_fin" / "dgraph_fin_pyg_data.pt",
}

for name, path in checks.items():
    status = "OK" if path.exists() else "MISSING"
    extra = ""
    if path.exists():
        extra = f" | {size_mb(path):.2f} MB"
    print(f"{status:<8} {name:<35} {path}{extra}")

print()
print("=" * 120)
print("FILE EXTENSION SUMMARY")
print("=" * 120)

ext_counts = {}
ext_sizes = {}

for p in DATA_ROOT.rglob("*"):
    if p.is_file():
        ext = p.suffix.lower() if p.suffix else "[no_ext]"
        ext_counts[ext] = ext_counts.get(ext, 0) + 1
        ext_sizes[ext] = ext_sizes.get(ext, 0.0) + size_mb(p)

for ext in sorted(ext_counts.keys()):
    print(f"{ext:<12} files={ext_counts[ext]:<8} size={ext_sizes[ext]:>10.2f} MB")

print()
print("=" * 120)
print("LIKELY USABLE DATASETS")
print("=" * 120)

usable = []

if (DATA_ROOT / "bitcoin_snap" / "bitcoin_alpha_edges.csv").exists():
    usable.append("Bitcoin-Alpha SNAP")

if (DATA_ROOT / "bitcoin_snap" / "bitcoin_otc_edges.csv").exists():
    usable.append("Bitcoin-OTC SNAP")

if (DATA_ROOT / "exports" / "fraud_yelp" / "fraud_yelp.bin").exists():
    usable.append("FraudYelp / YelpChi")

if (DATA_ROOT / "exports" / "fraud_amazon" / "fraud_amazon.bin").exists():
    usable.append("FraudAmazon")

if (DATA_ROOT / "exports" / "bitcoin_otc_dgl" / "bitcoin_otc_dgl.bin").exists():
    usable.append("Bitcoin-OTC DGL temporal snapshot")

if (DATA_ROOT / "pyg_reddit" / "reddit_pyg_data.pt").exists():
    usable.append("PyG Reddit raw graph")

if (DATA_ROOT / "pyg_dgraph_fin" / "dgraph_fin_pyg_data.pt").exists():
    usable.append("DGraph-Fin")

if (DATA_ROOT / "repos" / "GADBench").exists():
    gadbench_files = [p for p in (DATA_ROOT / "repos" / "GADBench").rglob("*") if p.is_file()]
    if len(gadbench_files) > 0:
        usable.append("GADBench repo/code")

if usable:
    for x in usable:
        print("OK -", x)
else:
    print("No usable datasets detected yet.")

print()
print("=" * 120)
print("DONE")
print("=" * 120)

GRAPH ANOMALY DATASETS AUDIT
DATA_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets
EXISTS: True

TOP-LEVEL SUMMARY
------------------------------------------------------------------------------------------------------------------------
bitcoin_snap                        dir   files=4        size=      2.02 MB
dgl_processed                       dir   files=3        size=    276.07 MB
dgl_raw                             dir   files=9        size=    795.97 MB
exports                             dir   files=23       size=    821.54 MB
pyg                                 dir   files=13       size=   7520.58 MB
repos                               dir   files=11       size=      7.38 MB

TREE, DEPTH=3
├── bitcoin_snap/  [4 files, 2.02 MB]
│   ├── bitcoin_alpha_edges.csv  [0.50 MB]
│   ├── bitcoin_otc_edges.csv  [1.00 MB]
│   ├── soc-sign-bitcoinalpha.csv.gz  [0.14 MB]
│   └── soc-sign-bitcoinotc.csv.gz  [0.38 MB]
├── dgl_processed/  [3 files, 276.07 MB]
│   ├── bitco

In [21]:
# ============================================================
# Inspect actual graph anomaly dataset structures
# Works with your current paths
# ============================================================

from pathlib import Path
import pandas as pd
import torch
import json
import os
import sys

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets")

PATHS = {
    "bitcoin_alpha": DATA_ROOT / "bitcoin_snap" / "bitcoin_alpha_edges.csv",
    "bitcoin_otc": DATA_ROOT / "bitcoin_snap" / "bitcoin_otc_edges.csv",

    "fraud_yelp_graph": DATA_ROOT / "exports" / "fraud_yelp" / "fraud_yelp.bin",
    "fraud_yelp_features": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_feature.pt",
    "fraud_yelp_labels": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_label.pt",
    "fraud_yelp_train_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_train_mask.pt",
    "fraud_yelp_val_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_val_mask.pt",
    "fraud_yelp_test_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_test_mask.pt",

    "fraud_amazon_graph": DATA_ROOT / "exports" / "fraud_amazon" / "fraud_amazon.bin",
    "fraud_amazon_features": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_feature.pt",
    "fraud_amazon_labels": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_label.pt",
    "fraud_amazon_train_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_train_mask.pt",
    "fraud_amazon_val_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_val_mask.pt",
    "fraud_amazon_test_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_test_mask.pt",

    "pyg_reddit": DATA_ROOT / "pyg" / "reddit" / "reddit_pyg_data.pt",
    "dgraph_fin": DATA_ROOT / "pyg" / "dgraph_fin" / "dgraph_fin_pyg_data.pt",

    "gadbench_reddit": DATA_ROOT / "repos" / "GADBench-master" / "datasets" / "reddit",
}

print("=" * 120)
print("PATH CHECK")
print("=" * 120)

for name, path in PATHS.items():
    status = "OK" if path.exists() else "MISSING"
    size = path.stat().st_size / 1024 / 1024 if path.exists() and path.is_file() else None
    if size is None:
        print(f"{status:<8} {name:<30} {path}")
    else:
        print(f"{status:<8} {name:<30} {path} | {size:.2f} MB")

PATH CHECK
OK       bitcoin_alpha                  C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_alpha_edges.csv | 0.50 MB
OK       bitcoin_otc                    C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_otc_edges.csv | 1.00 MB
OK       fraud_yelp_graph               C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp.bin | 128.95 MB
OK       fraud_yelp_features            C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_feature.pt | 5.61 MB
OK       fraud_yelp_labels              C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_label.pt | 0.35 MB
OK       fraud_yelp_train_mask          C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_train_m

In [22]:
# ============================================================
# Helpers
# ============================================================

def tensor_summary(x, name="tensor"):
    print(f"{name}:")
    print("  type:", type(x))
    
    if torch.is_tensor(x):
        print("  shape:", tuple(x.shape))
        print("  dtype:", x.dtype)
        print("  device:", x.device)
        
        if x.numel() > 0 and x.dtype in [
            torch.float16, torch.float32, torch.float64,
            torch.int8, torch.int16, torch.int32, torch.int64,
            torch.uint8, torch.bool
        ]:
            flat = x.flatten()
            print("  min:", flat.min().item())
            print("  max:", flat.max().item())
            
            if x.dtype in [torch.float16, torch.float32, torch.float64]:
                print("  mean:", flat.float().mean().item())
                print("  std:", flat.float().std().item())
            
            if x.numel() <= 30:
                print("  values:", flat.tolist())
            else:
                print("  first values:", flat[:10].tolist())
    else:
        print("  value:", x)


def label_summary(y, name="labels"):
    print(f"{name}:")
    if not torch.is_tensor(y):
        print("  not tensor:", type(y))
        return
    
    print("  shape:", tuple(y.shape))
    print("  dtype:", y.dtype)

    y_cpu = y.detach().cpu().flatten()
    vals, counts = torch.unique(y_cpu, return_counts=True)

    print("  classes:")
    total = len(y_cpu)
    for v, c in zip(vals.tolist(), counts.tolist()):
        print(f"    {v}: {c} ({c / total:.4f})")


def mask_summary(mask, name="mask"):
    print(f"{name}:")
    if not torch.is_tensor(mask):
        print("  not tensor:", type(mask))
        return
    
    mask = mask.detach().cpu().flatten()
    print("  shape:", tuple(mask.shape))
    print("  dtype:", mask.dtype)
    
    if mask.dtype == torch.bool:
        n = int(mask.sum().item())
        print(f"  true: {n} / {len(mask)} ({n / len(mask):.4f})")
    else:
        vals, counts = torch.unique(mask, return_counts=True)
        for v, c in zip(vals.tolist(), counts.tolist()):
            print(f"    {v}: {c}")


def csv_graph_summary(path, name="csv_graph", nrows_preview=5):
    print("=" * 120)
    print(name)
    print("=" * 120)
    print("path:", path)
    
    df = pd.read_csv(path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print()
    print("head:")
    display(df.head(nrows_preview))
    
    print()
    print("dtypes:")
    print(df.dtypes)
    
    if {"source", "target"}.issubset(df.columns):
        n_nodes = len(set(df["source"]).union(set(df["target"])))
        n_edges = len(df)
        print()
        print("graph approx:")
        print("  nodes:", n_nodes)
        print("  edges:", n_edges)
        print("  directed:", True)
        
        src_deg = df["source"].value_counts()
        dst_deg = df["target"].value_counts()
        
        print("  unique sources:", df["source"].nunique())
        print("  unique targets:", df["target"].nunique())
        print("  source degree mean:", src_deg.mean())
        print("  source degree max:", src_deg.max())
        print("  target degree mean:", dst_deg.mean())
        print("  target degree max:", dst_deg.max())
    
    if "rating" in df.columns:
        print()
        print("rating distribution:")
        print(df["rating"].describe())
        print("negative edges:", int((df["rating"] < 0).sum()))
        print("positive edges:", int((df["rating"] > 0).sum()))
        print("zero edges:", int((df["rating"] == 0).sum()))


def load_pt(path):
    return torch.load(path, map_location="cpu")

In [23]:
# ============================================================
# Inspect Bitcoin SNAP datasets
# ============================================================

csv_graph_summary(PATHS["bitcoin_alpha"], "Bitcoin-Alpha SNAP")
csv_graph_summary(PATHS["bitcoin_otc"], "Bitcoin-OTC SNAP")

Bitcoin-Alpha SNAP
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_alpha_edges.csv
shape: (24186, 4)
columns: ['source', 'target', 'rating', 'timestamp']

head:


,source,target,rating,timestamp
0,7188,1,10,1407470400
1,430,1,10,1376539200
2,3134,1,10,1369713600
3,3026,1,10,1350014400
4,3010,1,10,1347854400



dtypes:
source       int64
target       int64
rating       int64
timestamp    int64
dtype: object

graph approx:
  nodes: 3783
  edges: 24186
  directed: True
  unique sources: 3286
  unique targets: 3754
  source degree mean: 7.360316494217894
  source degree max: 490
  target degree mean: 6.442727757059137
  target degree max: 398

rating distribution:
count    24186.000000
mean         1.463946
std          2.903656
min        -10.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         10.000000
Name: rating, dtype: float64
negative edges: 1536
positive edges: 22650
zero edges: 0
Bitcoin-OTC SNAP
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_otc_edges.csv
shape: (35592, 4)
columns: ['source', 'target', 'rating', 'timestamp']

head:


,source,target,rating,timestamp
0,6,2,4,1.289242e+09
1,6,5,2,1.289242e+09
2,1,15,1,1.289243e+09
3,4,3,7,1.289245e+09
4,13,16,8,1.289254e+09



dtypes:
source         int64
target         int64
rating         int64
timestamp    float64
dtype: object

graph approx:
  nodes: 5881
  edges: 35592
  directed: True
  unique sources: 4814
  unique targets: 5858
  source degree mean: 7.393435812214375
  source degree max: 763
  target degree mean: 6.075793786275179
  target degree max: 535

rating distribution:
count    35592.000000
mean         1.012025
std          3.562094
min        -10.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         10.000000
Name: rating, dtype: float64
negative edges: 3563
positive edges: 32029
zero edges: 0


In [24]:
# ============================================================
# Inspect FraudYelp tensors
# ============================================================

print("=" * 120)
print("FraudYelp / YelpChi tensors")
print("=" * 120)

fy_x = load_pt(PATHS["fraud_yelp_features"])
fy_y = load_pt(PATHS["fraud_yelp_labels"])
fy_train = load_pt(PATHS["fraud_yelp_train_mask"])
fy_val = load_pt(PATHS["fraud_yelp_val_mask"])
fy_test = load_pt(PATHS["fraud_yelp_test_mask"])

tensor_summary(fy_x, "FraudYelp features X")
label_summary(fy_y, "FraudYelp labels y")
mask_summary(fy_train, "FraudYelp train_mask")
mask_summary(fy_val, "FraudYelp val_mask")
mask_summary(fy_test, "FraudYelp test_mask")

print()
print("Feature matrix sample:")
display(pd.DataFrame(fy_x[:5].numpy()))

FraudYelp / YelpChi tensors
FraudYelp features X:
  type: <class 'torch.Tensor'>
  shape: (45954, 32)
  dtype: torch.float32
  device: cpu
  min: 0.0
  max: 1.0
  mean: 0.6311110854148865
  std: 0.3221905529499054
  first values: [0.022375546395778656, 0.07049484550952911, 0.4286816418170929, 0.9999851584434509, 0.9999851584434509, 0.398456871509552, 0.8235922455787659, 0.4970250129699707, 0.9654573798179626, 0.15026336908340454]
FraudYelp labels y:
  shape: (45954,)
  dtype: torch.int64
  classes:
    0: 39277 (0.8547)
    1: 6677 (0.1453)
FraudYelp train_mask:
  shape: (45954,)
  dtype: torch.bool
  true: 32167 / 45954 (0.7000)
FraudYelp val_mask:
  shape: (45954,)
  dtype: torch.bool
  true: 4595 / 45954 (0.1000)
FraudYelp test_mask:
  shape: (45954,)
  dtype: torch.bool
  true: 9192 / 45954 (0.2000)

Feature matrix sample:


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.022376,0.070495,0.428682,0.999985,0.999985,0.398457,0.823592,0.497025,0.965457,0.150263,...,0.94806,0.867772,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512
1,0.024928,0.999985,0.999985,0.999985,0.999985,0.398457,0.999985,0.941257,0.217850,0.098019,...,0.94806,0.826367,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512
2,0.006173,0.070495,0.428682,0.999985,0.999985,0.398457,0.057230,0.151243,0.841056,1.000000,...,0.94806,0.802564,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512
3,0.017375,0.070495,0.428682,0.999985,0.999985,0.398457,0.999985,0.930203,0.025402,0.098019,...,0.94806,0.387121,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512
4,0.009081,0.999985,0.999985,0.999985,0.999985,0.398457,0.159374,0.697188,0.195489,1.000000,...,0.94806,0.246985,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512


In [25]:
# ============================================================
# Inspect FraudAmazon tensors
# ============================================================

print("=" * 120)
print("FraudAmazon tensors")
print("=" * 120)

fa_x = load_pt(PATHS["fraud_amazon_features"])
fa_y = load_pt(PATHS["fraud_amazon_labels"])
fa_train = load_pt(PATHS["fraud_amazon_train_mask"])
fa_val = load_pt(PATHS["fraud_amazon_val_mask"])
fa_test = load_pt(PATHS["fraud_amazon_test_mask"])

tensor_summary(fa_x, "FraudAmazon features X")
label_summary(fa_y, "FraudAmazon labels y")
mask_summary(fa_train, "FraudAmazon train_mask")
mask_summary(fa_val, "FraudAmazon val_mask")
mask_summary(fa_test, "FraudAmazon test_mask")

print()
print("Feature matrix sample:")
display(pd.DataFrame(fa_x[:5].numpy()))

FraudAmazon tensors
FraudAmazon features X:
  type: <class 'torch.Tensor'>
  shape: (11944, 25)
  dtype: torch.float32
  device: cpu
  min: -1.0
  max: 5525.0
  mean: 16.947477340698242
  std: 161.99424743652344
  first values: [1.0, 26.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
FraudAmazon labels y:
  shape: (11944,)
  dtype: torch.int64
  classes:
    0: 11123 (0.9313)
    1: 821 (0.0687)
FraudAmazon train_mask:
  shape: (11944,)
  dtype: torch.bool
  true: 6047 / 11944 (0.5063)
FraudAmazon val_mask:
  shape: (11944,)
  dtype: torch.bool
  true: 863 / 11944 (0.0723)
FraudAmazon test_mask:
  shape: (11944,)
  dtype: torch.bool
  true: 1729 / 11944 (0.1448)

Feature matrix sample:


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,1.0,26.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00,0.00,...,5.0,5.0,5.0,5.00,0.0,0.0,0.000000,1.0,13.0,1.0
1,4.0,17.0,0.0,1.0,1.0,0.0,2.0,0.0,0.25,0.25,...,4.0,5.0,2.0,3.75,0.0,3382.0,1.386294,0.0,45.0,1.0
2,2.0,15.0,0.0,0.0,0.0,2.0,0.0,0.0,0.00,0.00,...,4.0,4.0,4.0,4.00,0.0,0.0,0.000000,1.0,24.5,1.0
3,1.0,21.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00,0.00,...,5.0,5.0,5.0,5.00,0.0,0.0,0.000000,1.0,14.0,1.0
4,2.0,18.0,0.0,0.0,0.0,1.0,1.0,0.0,0.00,0.00,...,4.5,5.0,4.0,4.50,0.0,0.0,0.000000,1.0,18.5,1.0


In [26]:
# ============================================================
# Inspect exported edge relations for FraudYelp and FraudAmazon
# ============================================================

EDGE_PATHS = {
    "fraud_yelp_net_rsr": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rsr_review" / "fraud_yelp_review_net_rsr_review_edges.csv",
    "fraud_yelp_net_rtr": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rtr_review" / "fraud_yelp_review_net_rtr_review_edges.csv",
    "fraud_yelp_net_rur": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rur_review" / "fraud_yelp_review_net_rur_review_edges.csv",

    "fraud_amazon_net_upu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_upu_user" / "fraud_amazon_user_net_upu_user_edges.csv",
    "fraud_amazon_net_usu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_usu_user" / "fraud_amazon_user_net_usu_user_edges.csv",
    "fraud_amazon_net_uvu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_uvu_user" / "fraud_amazon_user_net_uvu_user_edges.csv",
}

print("=" * 120)
print("EDGE RELATION SUMMARY")
print("=" * 120)

edge_rows = []

for name, path in EDGE_PATHS.items():
    if not path.exists():
        print("[MISSING]", name, path)
        continue
    
    df = pd.read_csv(path, nrows=5)
    
    # Fast count rows without loading full CSV.
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        n_edges = sum(1 for _ in f) - 1
    
    full_cols = pd.read_csv(path, nrows=0).columns.tolist()
    
    edge_rows.append({
        "relation": name,
        "path": str(path),
        "n_edges": n_edges,
        "columns": full_cols,
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
    })
    
    print()
    print(name)
    print("path:", path)
    print("n_edges:", n_edges)
    print("columns:", full_cols)
    display(df)

edge_summary_df = pd.DataFrame(edge_rows)
display(edge_summary_df)

EDGE RELATION SUMMARY

fraud_yelp_net_rsr
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\edges_review_net_rsr_review\fraud_yelp_review_net_rsr_review_edges.csv
n_edges: 6805486
columns: ['source', 'target', 'src_type', 'edge_type', 'dst_type']


,source,target,src_type,edge_type,dst_type
0,2,0,review,net_rsr,review
1,6702,0,review,net_rsr,review
2,4,1,review,net_rsr,review
3,0,2,review,net_rsr,review
4,6702,2,review,net_rsr,review



fraud_yelp_net_rtr
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\edges_review_net_rtr_review\fraud_yelp_review_net_rtr_review_edges.csv
n_edges: 1147232
columns: ['source', 'target', 'src_type', 'edge_type', 'dst_type']


,source,target,src_type,edge_type,dst_type
0,5,3,review,net_rtr,review
1,6703,4,review,net_rtr,review
2,3,5,review,net_rtr,review
3,7,6,review,net_rtr,review
4,9,6,review,net_rtr,review



fraud_yelp_net_rur
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\edges_review_net_rur_review\fraud_yelp_review_net_rur_review_edges.csv
n_edges: 98630
columns: ['source', 'target', 'src_type', 'edge_type', 'dst_type']


...


fraud_amazon_net_usu
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_amazon\edges_user_net_usu_user\fraud_amazon_user_net_usu_user_edges.csv
n_edges: 7132958
columns: ['source', 'target', 'src_type', 'edge_type', 'dst_type']


,source,target,src_type,edge_type,dst_type
0,74,0,user,net_usu,user
1,139,0,user,net_usu,user
2,317,0,user,net_usu,user
3,725,0,user,net_usu,user
4,1968,0,user,net_usu,user



fraud_amazon_net_uvu
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_amazon\edges_user_net_uvu_user\fraud_amazon_user_net_uvu_user_edges.csv
n_edges: 2073474
columns: ['source', 'target', 'src_type', 'edge_type', 'dst_type']


,source,target,src_type,edge_type,dst_type
0,2486,0,user,net_uvu,user
1,4857,0,user,net_uvu,user
2,5009,0,user,net_uvu,user
3,5263,0,user,net_uvu,user
4,5610,0,user,net_uvu,user


,relation,path,n_edges,columns,size_mb
0,fraud_yelp_net_rsr,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,6805486,"[source, target, src_type, edge_type, dst_type]",225.34
1,fraud_yelp_net_rtr,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,1147232,"[source, target, src_type, edge_type, dst_type]",38.02
2,fraud_yelp_net_rur,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,98630,"[source, target, src_type, edge_type, dst_type]",3.22
3,fraud_amazon_net_upu,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,351216,"[source, target, src_type, edge_type, dst_type]",9.72
4,fraud_amazon_net_usu,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,7132958,"[source, target, src_type, edge_type, dst_type]",198.22
5,fraud_amazon_net_uvu,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,2073474,"[source, target, src_type, edge_type, dst_type]",57.51


In [27]:
# ============================================================
# Inspect DGL graphs
# ============================================================

try:
    import dgl
    from dgl.data.utils import load_graphs

    def inspect_dgl_bin(path, name):
        print("=" * 120)
        print(name)
        print("=" * 120)
        print("path:", path)

        graphs, labels = load_graphs(str(path))
        print("num graphs:", len(graphs))
        print("labels object:", labels)

        g = graphs[0]
        print(g)

        print()
        print("node types:", g.ntypes)
        print("edge types:", g.etypes)
        print("canonical etypes:", g.canonical_etypes)

        print()
        print("node data:")
        for ntype in g.ntypes:
            print(" ntype:", ntype)
            for key, val in g.nodes[ntype].data.items():
                print(f"   {key}: shape={tuple(val.shape)}, dtype={val.dtype}")

        print()
        print("edge data:")
        for etype in g.canonical_etypes:
            print(" etype:", etype)
            for key, val in g.edges[etype].data.items():
                print(f"   {key}: shape={tuple(val.shape)}, dtype={val.dtype}")

        return g

    g_yelp = inspect_dgl_bin(PATHS["fraud_yelp_graph"], "DGL FraudYelp graph")
    g_amazon = inspect_dgl_bin(PATHS["fraud_amazon_graph"], "DGL FraudAmazon graph")

except Exception as e:
    print("[ERROR] Could not inspect DGL bins")
    print(repr(e))

DGL FraudYelp graph
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp.bin
num graphs: 1
labels object: {}
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])

node types: ['review']
edge types: ['net_rsr', 'net_rtr', 'net_rur']
canonical etypes: [('review', 'net_rsr', 'review'), ('review', 'net_rtr', 'review'), ('review', 'net_rur', 'review')]

node data:
 ntype: review
   val_mask: shape=(45954,), dtype=torch.uint8
   label: shape=(45954,), dtype=torch.int64
   feature: shape=(45954, 32), dtype=torch.float32
   train_mask: shape=(45954,), dtype=torch.uint8
   test_mask: shape=(45954,), dtype=torch.uint8

edge data:
 etype: ('review', 'net_rsr', 'review')
 etype: ('review', 'net_rtr', 'review')
 e

In [28]:
# ============================================================
# Inspect PyG Reddit
# ============================================================

print("=" * 120)
print("PyG Reddit")
print("=" * 120)

reddit_path = PATHS["pyg_reddit"]

if reddit_path.exists():
    reddit_data = torch.load(reddit_path, map_location="cpu")
    print(reddit_data)
    print()
    print("keys:", list(reddit_data.keys()) if hasattr(reddit_data, "keys") else None)

    for key in reddit_data.keys():
        value = reddit_data[key]
        if torch.is_tensor(value):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
        else:
            print(f"{key}: {type(value)}")

    if hasattr(reddit_data, "y") and reddit_data.y is not None:
        label_summary(reddit_data.y, "PyG Reddit labels")

    if hasattr(reddit_data, "train_mask") and reddit_data.train_mask is not None:
        mask_summary(reddit_data.train_mask, "PyG Reddit train_mask")
    if hasattr(reddit_data, "val_mask") and reddit_data.val_mask is not None:
        mask_summary(reddit_data.val_mask, "PyG Reddit val_mask")
    if hasattr(reddit_data, "test_mask") and reddit_data.test_mask is not None:
        mask_summary(reddit_data.test_mask, "PyG Reddit test_mask")

else:
    print("[MISSING]", reddit_path)

PyG Reddit
Data(x=[232965, 602], edge_index=[2, 114615892], y=[232965], train_mask=[232965], val_mask=[232965], test_mask=[232965])

keys: ['val_mask', 'y', 'x', 'edge_index', 'train_mask', 'test_mask']
val_mask: shape=(232965,), dtype=torch.bool
y: shape=(232965,), dtype=torch.int64
x: shape=(232965, 602), dtype=torch.float32
edge_index: shape=(2, 114615892), dtype=torch.int64
train_mask: shape=(232965,), dtype=torch.bool
test_mask: shape=(232965,), dtype=torch.bool
PyG Reddit labels:
  shape: (232965,)
  dtype: torch.int64
  classes:
    0: 13101 (0.0562)
    1: 3550 (0.0152)
    2: 3302 (0.0142)
    3: 15181 (0.0652)
    4: 2322 (0.0100)
    5: 3597 (0.0154)
    6: 3952 (0.0170)
    7: 2138 (0.0092)
    8: 11187 (0.0480)
    9: 2246 (0.0096)
    10: 4928 (0.0212)
    11: 2964 (0.0127)
    12: 1696 (0.0073)
    13: 2731 (0.0117)
    14: 4854 (0.0208)
    15: 28272 (0.1214)
    16: 1003 (0.0043)
    17: 2639 (0.0113)
    18: 13999 (0.0601)
    19: 10308 (0.0442)
    20: 1596 (0.0069)


In [29]:
# ============================================================
# Inspect DGraph-Fin PyG
# ============================================================

print("=" * 120)
print("DGraph-Fin")
print("=" * 120)

dgraph_path = PATHS["dgraph_fin"]

if dgraph_path.exists():
    dgraph_data = torch.load(dgraph_path, map_location="cpu")
    print(dgraph_data)
    print()
    print("keys:", list(dgraph_data.keys()) if hasattr(dgraph_data, "keys") else None)

    for key in dgraph_data.keys():
        value = dgraph_data[key]
        if torch.is_tensor(value):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
        else:
            print(f"{key}: {type(value)}")

    if hasattr(dgraph_data, "y") and dgraph_data.y is not None:
        label_summary(dgraph_data.y, "DGraph-Fin labels")

    if hasattr(dgraph_data, "train_mask") and dgraph_data.train_mask is not None:
        mask_summary(dgraph_data.train_mask, "DGraph-Fin train_mask")
    if hasattr(dgraph_data, "val_mask") and dgraph_data.val_mask is not None:
        mask_summary(dgraph_data.val_mask, "DGraph-Fin val_mask")
    if hasattr(dgraph_data, "test_mask") and dgraph_data.test_mask is not None:
        mask_summary(dgraph_data.test_mask, "DGraph-Fin test_mask")

else:
    print("[MISSING]", dgraph_path)

DGraph-Fin
Data(x=[3700550, 17], edge_index=[2, 4300999], y=[3700550], edge_type=[4300999], edge_time=[4300999], train_mask=[3700550], val_mask=[3700550], test_mask=[3700550])

keys: ['val_mask', 'y', 'edge_type', 'edge_time', 'x', 'edge_index', 'train_mask', 'test_mask']
val_mask: shape=(3700550,), dtype=torch.bool
y: shape=(3700550,), dtype=torch.int64
edge_type: shape=(4300999,), dtype=torch.int64
edge_time: shape=(4300999,), dtype=torch.int64
x: shape=(3700550, 17), dtype=torch.float32
edge_index: shape=(2, 4300999), dtype=torch.int64
train_mask: shape=(3700550,), dtype=torch.bool
test_mask: shape=(3700550,), dtype=torch.bool
DGraph-Fin labels:
  shape: (3700550,)
  dtype: torch.int64
  classes:
    0: 1210092 (0.3270)
    1: 15509 (0.0042)
    2: 1620851 (0.4380)
    3: 854098 (0.2308)
DGraph-Fin train_mask:
  shape: (3700550,)
  dtype: torch.bool
  true: 857899 / 3700550 (0.2318)
DGraph-Fin val_mask:
  shape: (3700550,)
  dtype: torch.bool
  true: 183862 / 3700550 (0.0497)
DGraph

In [30]:
# ============================================================
# Inspect GADBench Reddit file
# ============================================================

print("=" * 120)
print("GADBench Reddit")
print("=" * 120)

gad_path = PATHS["gadbench_reddit"]

if gad_path.exists():
    print("path:", gad_path)
    print("size MB:", gad_path.stat().st_size / 1024 / 1024)

    # Try different loading strategies.
    loaded = None
    
    try:
        loaded = torch.load(gad_path, map_location="cpu")
        print("[LOAD] torch.load OK")
    except Exception as e:
        print("[LOAD] torch.load failed:", repr(e))
    
    if loaded is None:
        try:
            import pickle
            with open(gad_path, "rb") as f:
                loaded = pickle.load(f)
            print("[LOAD] pickle OK")
        except Exception as e:
            print("[LOAD] pickle failed:", repr(e))
    
    if loaded is not None:
        print("object type:", type(loaded))

        if isinstance(loaded, dict):
            print("dict keys:", loaded.keys())
            for k, v in loaded.items():
                if torch.is_tensor(v):
                    print(f"{k}: tensor shape={tuple(v.shape)}, dtype={v.dtype}")
                elif hasattr(v, "shape"):
                    print(f"{k}: shape={v.shape}, type={type(v)}")
                else:
                    print(f"{k}: type={type(v)}")
        
        elif hasattr(loaded, "keys"):
            print("keys:", list(loaded.keys()))
            for k in loaded.keys():
                v = loaded[k]
                if torch.is_tensor(v):
                    print(f"{k}: tensor shape={tuple(v.shape)}, dtype={v.dtype}")
                elif hasattr(v, "shape"):
                    print(f"{k}: shape={v.shape}, type={type(v)}")
                else:
                    print(f"{k}: type={type(v)}")
        
        else:
            print(loaded)

else:
    print("[MISSING]", gad_path)

GADBench Reddit
path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\repos\GADBench-master\datasets\reddit
size MB: 7.244559288024902
[LOAD] torch.load failed: UnpicklingError("invalid load key, '?'.")
[LOAD] pickle failed: UnpicklingError("invalid load key, '?'.")


In [31]:
# ============================================================
# Build final usable dataset table
# ============================================================

rows = []

def add_dataset(name, kind, path, has_labels=None, has_features=None, notes=""):
    rows.append({
        "dataset": name,
        "kind": kind,
        "exists": path.exists(),
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() and path.is_file() else None,
        "has_labels": has_labels,
        "has_features": has_features,
        "notes": notes,
    })

add_dataset(
    "Bitcoin-Alpha",
    "SNAP signed graph CSV",
    PATHS["bitcoin_alpha"],
    has_labels=False,
    has_features=False,
    notes="No node anomaly labels; use for signed-graph shortcut/stress-test features."
)

add_dataset(
    "Bitcoin-OTC",
    "SNAP signed graph CSV",
    PATHS["bitcoin_otc"],
    has_labels=False,
    has_features=False,
    notes="No node anomaly labels; use for signed-graph shortcut/stress-test features."
)

add_dataset(
    "FraudYelp / YelpChi",
    "DGL heterograph fraud benchmark",
    PATHS["fraud_yelp_graph"],
    has_labels=True,
    has_features=True,
    notes="Good main anomaly benchmark."
)

add_dataset(
    "FraudAmazon",
    "DGL heterograph fraud benchmark",
    PATHS["fraud_amazon_graph"],
    has_labels=True,
    has_features=True,
    notes="Good main anomaly benchmark."
)

add_dataset(
    "PyG Reddit",
    "PyG raw node classification graph",
    PATHS["pyg_reddit"],
    has_labels=True,
    has_features=True,
    notes="Not anomaly by default; use for synthetic/structural anomalies."
)

add_dataset(
    "DGraph-Fin",
    "PyG financial anomaly graph",
    PATHS["dgraph_fin"],
    has_labels=True,
    has_features=True,
    notes="Good large-scale financial anomaly benchmark."
)

add_dataset(
    "GADBench Reddit",
    "GADBench anomaly example",
    PATHS["gadbench_reddit"],
    has_labels=True,
    has_features=True,
    notes="Likely useful as small ready anomaly benchmark; inspect object keys first."
)

summary_df = pd.DataFrame(rows)
display(summary_df)

,dataset,kind,exists,path,size_mb,has_labels,has_features,notes
0,Bitcoin-Alpha,SNAP signed graph CSV,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,0.50,False,False,No node anomaly labels; use for signed-graph s...
1,Bitcoin-OTC,SNAP signed graph CSV,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,1.00,False,False,No node anomaly labels; use for signed-graph s...
2,FraudYelp / YelpChi,DGL heterograph fraud benchmark,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,128.95,True,True,Good main anomaly benchmark.
3,FraudAmazon,DGL heterograph fraud benchmark,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,147.11,True,True,Good main anomaly benchmark.
4,PyG Reddit,PyG raw node classification graph,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,2286.34,True,True,Not anomaly by default; use for synthetic/stru...
5,DGraph-Fin,PyG financial anomaly graph,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,410.06,True,True,Good large-scale financial anomaly benchmark.
6,GADBench Reddit,GADBench anomaly example,True,C:\Users\ivan\WORK\workshops\ICDM\Ablation\gra...,7.24,True,True,Likely useful as small ready anomaly benchmark...
